# TextZoom baseline check

1. Reproduce the **BICUBIC** rows of Table 4 (arXiv:2005.03341) with the migrated code. This validates data, recognizers and the migration.
2. Evaluate a **community-trained TSRN** checkpoint as a possible pruning starting point.
   2b. Train a **clean TSRN baseline** with the original recipe on the 95% training split (resumable; A100 recommended).
3. Time a few minutes of **TSRN** training to estimate the cost of the full 500 epochs.
4. Profile **TSRN** inference: time share of GRUs vs convolutions on GPU and CPU (needs section 2 for the checkpoint path).
5. **Pruning pilot**: one 20% pruning step on TSRN's convolutions, short fine-tune, then accuracy, PSNR, size and speed (needs section 2).
6. **RQ1 trajectory**: prune step by step to 87.5% of the channels, recording PSNR and OCR after every step (resumable).
7. **RQ1/RQ2 figures**: PSNR change vs OCR change, cost vs speed, and both stop rules applied to the recorded curve.
8. **Same-session latency** of every trajectory model, for the paper.
9. **Re-score** every model with the three readers, image by image (steadier OCR stop rule, McNemar tests).
10. **Example figure**: images where pruning changes what CRNN reads.
11. **Reading-aware pruning pilot** (journal): image-loss vs reading-loss channel scoring and fine-tuning at one 50% cut.
12. **Controls** for the pilot: uncut model with image-only and reading-aware fine-tuning, and image-only pruning with the same training time.
13. **Confirmation**: uncut model with reading-aware fine-tuning for a fixed 20 epochs (price of pruning at equal training time).
14. **Fair 2 x 2**: uncut vs cut 50%, image vs reading fine-tuning, all trained 20 epochs keeping the last one.
15. **87.5% cut**, image vs reading fine-tuning.
16. **GRU pruning** (width): GRU units removed, with and without the convolution cut.
17. **Depth vs width speed test** on random weights: which cuts make one image really faster.
18. **Depth pruning**: whole blocks or GRUs removed (chosen by reading), then fine-tuned.
19. **Pruned vs trained from scratch**: the same 2-block TSRN trained from scratch, and the pruned one fine-tuned longer.
20. **LEMMA as a teacher**: LEMMA's released model run in our pipeline (reading, size, speed), then T fine-tuned with LEMMA as its teacher (kill test).
21. **LEMMA with a small guidance network**: LEMMA's text-reading branch replaced by a small network trained to copy it (kill test).
22. **Checks before a small reader for LEMMA**: what LEMMA's reader costs part by part, and what reading it must keep (no training).

Runtime → Change runtime type → **T4 GPU** before running (**A100** for section 2b if available; latency in section 8 always on a T4).

**Results on Drive** (`MyDrive/textzoom/`):
```
data/  aster.pth.tar  moran.pth  crnn.pth  tsrn_community.pth   inputs, downloaded once
experiments/
  baseline/<run name>/         section 2b (checkpoints/final.pth = the clean baseline)
  pilot/<run name>/            section 5
  rq1_trajectory/<run name>/   sections 6-8 (each run folder has a README.md listing its files)
```
Run names describe the settings, e.g. `tsrn-clean_conv-only_taylor_linear-7steps-to88pct_round8_ft20ep-es3_kd0.1_select-psnr_seed0` (runs from the community checkpoint start with `tsrn_conv-only_`).

In [ ]:
# Closest stack to the SwinIR pruning project that installs on Python 3.13.
!nvidia-smi && python --version
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0

In [ ]:
REPO = '/content/TextZoomPruning'
%cd /content
# Clone the first time; afterwards update, so a reused runtime never runs old code.
![ -d {REPO}/.git ] && git -C {REPO} pull || git clone -b migrate-torch2 https://github.com/Maouhoub/TextZoomPruning.git
%cd {REPO}
!pip install -q -r requirements.txt

import torch, torchvision, numpy
print(torch.__version__, torchvision.__version__, numpy.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Downloads are kept on Drive so later sessions skip them.
from google.colab import drive
drive.mount('/content/drive')

import os
STORE = '/content/drive/MyDrive/textzoom'
os.makedirs(STORE, exist_ok=True)

In [ ]:
%cd {STORE}
# TextZoom LMDB (official Google Drive folder from the TextZoom README)
![ -f .data_done ] || (gdown --folder https://drive.google.com/drive/folders/1WRVy-fC_KrembPkaI68uqQ9wyaptibMh -O data && touch .data_done)
# Pretrained recognizers, used as-is
![ -f aster.pth.tar ] || wget -q -O aster.pth.tar https://github.com/ayumiymk/aster.pytorch/releases/download/v1.0/demo.pth.tar
![ -f moran.pth ] || gdown 1IDvT51MXKSseDq3X57uPjOzeSYI09zip -O moran.pth
![ -f crnn.pth ] || wget -q -O crnn.pth "https://www.dropbox.com/s/dboqjk20qjkpta3/crnn.pth?dl=1"
!ls -la {STORE}

# LMDB reads from the Drive mount are slow, so work on a local copy.
!rm -rf /content/data && cp -r {STORE}/data /content/data
!find /content/data -name data.mdb

In [ ]:
# Point the config at the data and recognizer weights.
import glob, yaml

lmdb_dirs = sorted(os.path.dirname(p) for p in glob.glob('/content/data/**/data.mdb', recursive=True))
by_name = {os.path.basename(d): d for d in lmdb_dirs}
print(by_name)
TEST = {split: by_name[split] for split in ('easy', 'medium', 'hard')}

cfg_path = f'{REPO}/src/config/super_resolution.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['TRAIN']['train_data_dir'] = [by_name['train1'], by_name['train2']]
cfg['TRAIN']['VAL']['val_data_dir'] = [TEST['hard'], TEST['easy'], TEST['medium']]
cfg['TRAIN']['VAL']['rec_pretrained'] = f'{STORE}/aster.pth.tar'
cfg['TRAIN']['VAL']['moran_pretrained'] = f'{STORE}/moran.pth'
cfg['TRAIN']['VAL']['crnn_pretrained'] = f'{STORE}/crnn.pth'
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
print(open(cfg_path).read())

## 1. BICUBIC sanity check (no training)

Our accuracy should be within ~1 point of the paper. If ASTER/MORAN are off by more, the `align_corners` choice is the first suspect.

In [ ]:
import ast, subprocess

SPLITS = ('easy', 'medium', 'hard')
SPLIT_SIZES = (1619, 1411, 1343)  # Table 1, used for the paper's weighted average
PAPER_BICUBIC = {'aster': (64.7, 42.4, 31.2), 'moran': (60.6, 37.9, 30.8), 'crnn': (36.4, 21.1, 21.1)}  # Table 4
PAPER_TSRN = {'aster': (75.1, 56.3, 40.1), 'moran': (70.1, 53.3, 37.9), 'crnn': (52.5, 38.2, 31.4)}  # Table 4
PAPER_BICUBIC_PSNR = (22.35, 18.98, 19.39)  # Table 11


def weighted_average(per_split):
    return sum(a * n for a, n in zip(per_split, SPLIT_SIZES)) / sum(SPLIT_SIZES)


def evaluate(model_args, paper_acc):
    """Run main.py --test for every recognizer and split; print accuracy next to the paper."""
    results = {}
    for rec, paper_row in paper_acc.items():
        ours_row = []
        for split, paper in zip(SPLITS, paper_row):
            run = subprocess.run(['python', 'main.py', '--test', '--rec', rec, '--batch_size', '256',
                                  '--test_data_dir', TEST[split], *model_args],
                                 cwd=f'{REPO}/src', capture_output=True, text=True)
            lines = run.stdout.strip().splitlines()
            if run.returncode != 0 or not lines or not lines[-1].startswith('{'):
                print(run.stdout[-2000:], run.stderr[-4000:])
                raise RuntimeError(f'{rec}/{split} failed')
            result = ast.literal_eval(lines[-1])
            results[(rec, split)] = result
            acc = result['accuracy'][split] * 100
            ours_row.append(acc)
            print(f'{rec:6s} {split:7s} acc {acc:5.1f} (paper {paper:5.1f})  psnr {result["psnr_avg"]:.2f}')
        print(f'{rec:6s} average acc {weighted_average(ours_row):5.1f} (paper {weighted_average(paper_row):5.1f})\n')
    return results


bicubic_results = evaluate(['--arch', 'bicubic'], PAPER_BICUBIC)
print('paper PSNR easy/medium/hard:', PAPER_BICUBIC_PSNR)

## 2. Community TSRN checkpoint

A trained TSRN shared by a TextZoom user (yustiks) on Google Drive; reported ASTER 73.3 / 56.2 / 39.2 against the paper's 75.1 / 56.3 / 40.1.

The file comes from an unknown source, so it is loaded with `weights_only=True`: a checkpoint containing anything other than tensors and plain values is refused instead of executed. If it loads and scores close to the paper, use it as the pruning starting point and report its source together with the numbers measured here.

In [ ]:
CKPT = f'{STORE}/tsrn_community.pth'
![ -f {CKPT} ] || gdown 1j-g17V5kBmqS8giWNZuHe2d_GefAepCs -O {CKPT}

ckpt = torch.load(CKPT, map_location='cpu', weights_only=True)
print('top-level keys:', list(ckpt.keys()))
print('info:', ckpt.get('info'))
state = ckpt['state_dict_G']
print(len(state), 'tensors, first ones:', list(state)[:4])

# The STN (stn_head, tps) only runs during training; BatchNorm running statistics are not weights.
def is_inference_weight(name):
    return not name.startswith(('stn_head', 'tps')) and 'running_' not in name and 'num_batches_tracked' not in name

print(f"weights used at inference: {sum(v.numel() for k, v in state.items() if is_inference_weight(k)):,}")
print(f"all weights incl. STN:     {sum(v.numel() for k, v in state.items() if 'running_' not in k and 'num_batches_tracked' not in k):,}")

In [ ]:
tsrn_results = evaluate(['--arch', 'tsrn', '--STN', '--mask', '--resume', CKPT], PAPER_TSRN)

## 2b. Clean TSRN baseline (own training, resumable)

The community checkpoint was trained on the whole training set, so it has already seen our 868 validation images, and the original code keeps the checkpoint that is best on the test sets. This trains TSRN with the original recipe (STN + mask, MSE + gradient prior loss, Adam lr 1e-3 / beta1 0.5, batch 512, 500 epochs) on the other 95% only, and keeps the **final** weights: nothing is selected on the test sets, which are evaluated once at the end.

- **A100** (40 GB): batch 512 fits, about 2-3 hours. **T4**: the cell adds `--micro_batch 256` (still one update per 512 images), about 8-9 hours.
- Validation (PSNR + ASTER/MORAN/CRNN on the 868 images) every 5 epochs, recorded in `history.csv`.
- Saved in `experiments/baseline/<run name>/`; the baseline used by section 6 is `checkpoints/final.pth`.

**If Colab disconnects:** reconnect, run the setup cells (install → config), then this cell again. It continues from the last saved state (at most 5 epochs are redone), also on another GPU type.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
# Batch 512 does not fit on a 16 GB T4: smaller GPUs split each batch in pieces of 256 (same update).
big_gpu = torch.cuda.get_device_properties(0).total_memory > 30e9
TRAIN_ARGS = f'--workers {min(8, os.cpu_count())}' + ('' if big_gpu else ' --micro_batch 256')
print(torch.cuda.get_device_name(0), '|', TRAIN_ARGS)

%cd {REPO}/src
!python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} {TRAIN_ARGS}

In [ ]:
# Training curve of the clean baseline (works while training runs too); saved in its figures/ folder.
import matplotlib.pyplot as plt
import pandas as pd

%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
CLEAN_DIR = CLEAN_DIR[-1]
history = pd.read_csv(f'{CLEAN_DIR}/history.csv')
val = history[history['val_psnr'].notna()]
READER_COLORS = {'aster': ('ASTER', '#2a78d6'), 'moran': ('MORAN', '#eb6834'), 'crnn': ('CRNN', '#1baf7a')}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)
axes[0].plot(history['epoch'], history['loss'], color='#0b0b0b')
axes[0].set_title('a) Training loss', loc='left')
axes[1].plot(val['epoch'], val['val_psnr'], color='#0b0b0b', marker='o', markersize=3)
axes[1].set_title('b) Validation PSNR (dB)', loc='left')
for rec, (label, color) in READER_COLORS.items():
    axes[2].plot(val['epoch'], val[f'val_{rec}'], color=color, label=label)
axes[2].plot(val['epoch'], val['val_ocr_average'], color='#52514e', linestyle='--', label='average')
axes[2].set_title('c) Validation accuracy (%)', loc='left')
axes[2].legend(frameon=False)
for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(True, color='#e4e3df')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
for ext in ('png', 'pdf'):
    fig.savefig(f'{CLEAN_DIR}/figures/training_curve.{ext}', dpi=200)
plt.show()
print(val.tail(3).to_string(index=False))

## 3. Time TSRN training (15 minutes)

Prints every 10 iterations. One epoch is 67 iterations at batch 256 (17,367 images). Estimate: seconds per iteration × 67 × 500 epochs. The paper's batch of 512 was split over 4 GPUs (128 each), so 512 does not fit on one T4. Validation (ASTER on the 3 test sets) runs every 200 iterations and is included. If you get an out-of-memory error, use `--batch_size 128`.

In [ ]:
cfg = yaml.safe_load(open(cfg_path))
cfg['TRAIN']['displayInterval'] = 10
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)

%cd {REPO}/src
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True timeout 15m python main.py --arch tsrn --batch_size 256 --STN --mask --gradient --vis_dir tsrn_timing

## 4. Profile TSRN inference (GRU vs convolutions)

Median latency per batch, then the share of op time per category (GPU: kernel time; CPU: CPU time), at batch 1 and 64, on GPU and on CPU with 1 thread and all threads. If GRUs take most of the time, pruning convolutions alone will not make TSRN much faster. A low "GPU busy" share means the GPU mostly waits for kernel launches, so fewer channels barely help either. Takes a few minutes, mostly the CPU runs.

In [ ]:
%cd {REPO}/src
!python profile_tsrn.py --resume {CKPT}

## 5. Pruning pilot (one step, convolutions only)

1. Holds out 5% of the training set as a validation split (saved as `val_split.json`).
2. Removes 20% of the channels of every prunable convolution group (Taylor importance). GRUs, the trunk width, the input and the 4-channel output are protected.
3. Checks the output is still a 4 × 32 × 128 image.
4. Fine-tunes for 3 epochs with the unpruned model as teacher; the best epoch is picked on validation PSNR.
5. Reports params, MACs, validation CRNN accuracy and PSNR, test accuracy with all three readers, and latency, before and after.

Results are saved to Drive. Takes roughly 30 minutes, mostly the test evaluations; add `--skip_test` for a quick validation-only run.

In [ ]:
import shutil

EXPERIMENTS = f'{STORE}/experiments'
PILOT_DIR = f'{EXPERIMENTS}/pilot/tsrn_conv-only_taylor_one-step-20pct_round4_ft3ep_kd0.1_select-psnr_seed0'
# Earlier sessions saved the pilot to textzoom/pilot_ratio0.2: move it under experiments/.
if os.path.isdir(f'{STORE}/pilot_ratio0.2') and not os.path.exists(PILOT_DIR):
    os.makedirs(os.path.dirname(PILOT_DIR), exist_ok=True)
    shutil.move(f'{STORE}/pilot_ratio0.2', PILOT_DIR)

%cd {REPO}/src
!python prune_tsrn_pilot.py --resume {CKPT} --out_dir {PILOT_DIR}

## 6. RQ1 trajectory (resumable)

Seven steps of 8 channels per group (64 → 56 → … → 8, i.e. 12.5% more removed at every step), each followed by fine-tuning with the unpruned TSRN as teacher. After every step: size, MACs, latency, validation PSNR and CRNN accuracy, and test accuracy with ASTER, MORAN and CRNN. No stop rule, so the whole curve is recorded.

Settings, each combination in its own folder:
- `BASELINE`: `'clean'` = our own TSRN from section 2b (folder names start with `tsrn-clean_`); `'community'` = the community checkpoint (the first runs, folder names start with `tsrn_`).
- `RUN`: `'short'` = 3 fine-tuning epochs per step (about 1.5-3 hours on a T4); `'main'` = up to 20 epochs per step, stopping early once validation PSNR has not improved by more than 0.01 dB for 3 epochs (as in Chapter 1; about 2.5-4 hours on a T4).
- `SEED`: seed of the pruning and fine-tuning; run 0, 1 and 2 for the paper. The validation images stay the same for every seed (split seed 0, as in section 2b).

**If Colab disconnects:** reconnect, run the setup cells (install, clone, Drive, downloads/data copy, config), then run this cell again with the same settings. It continues after the last finished step; an interrupted step is redone from the previous one.

In [ ]:
BASELINE = 'clean'  # 'clean' (section 2b) or 'community'
RUN = 'main'        # 'short' or 'main'
SEED = 0            # 0, 1, 2
RUN_ARGS = {'short': '', 'main': '--epochs 20 --patience 3 --min_delta 0.01'}[RUN]

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = {'clean': f'{CLEAN_DIR[-1]}/checkpoints/final.pth',
                 'community': f'{STORE}/tsrn_community.pth'}[BASELINE]
assert os.path.exists(BASELINE_CKPT), f'{BASELINE_CKPT} not found (finish section 2b first)'
!python prune_tsrn_trajectory.py --resume {BASELINE_CKPT} --experiments_dir {EXPERIMENTS} --seed {SEED} {RUN_ARGS}

## 7. RQ1/RQ2 figures and stop rules

a) validation PSNR change, b) test accuracy change per reader, c) MACs and latency relative to the unpruned model (1.0 = unpruned). Saved as PNG and PDF in each run's `figures/` folder; panel c uses `latency_same_session.csv` once section 8 has run.

Both stop rules are then applied to the recorded **validation** curve: each keeps the last step before its validation drop first exceeds the limit. With 868 validation images, CRNN accuracy has a standard error of about 1.6 points, so read the OCR rule with that in mind.

In [ ]:
import glob
import matplotlib.pyplot as plt
import pandas as pd

EXPERIMENTS = f'{STORE}/experiments'

READERS = {'aster': ('ASTER', '#2a78d6'), 'moran': ('MORAN', '#eb6834'), 'crnn': ('CRNN', '#1baf7a')}
LATENCY = {'gpu_b1': ('GPU, 1 image', '#6250d6', 'o'), 'gpu_b64': ('GPU, 64 images', '#008300', '^'),
           'cpu1_b1': ('CPU 1 thread, 1 image', '#e34948', 'D')}
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
PSNR_LIMIT, OCR_LIMIT = 0.2, 1.0  # Chapter 1 PSNR stop rule (dB) and the OCR rule of RQ2 (points)


def style(ax, title, ylabel):
    ax.set_title(title, loc='left', color=INK, fontsize=11)
    ax.set_xlabel('channels removed (%)', color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=MUTED)


def limit_line(ax, y, text, x_right):
    ax.axhline(y, color=MUTED, linestyle='--', linewidth=1)
    ax.annotate(text, (x_right, y), xytext=(0, 3), textcoords='offset points', ha='right', va='bottom',
                color=MUTED, fontsize=9)


def kept_step(df, column, limit):
    """Step a stop rule keeps: the last one before the validation drop first exceeds `limit`."""
    drop = df[column].iloc[0] - df[column]
    over = [i for i, d in enumerate(drop) if d > limit]
    return int(df['step'].iloc[over[0] - 1 if over else -1])


for csv_path in sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/*/trajectory.csv')):
    run_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    base, x = df.iloc[0], df['removed_pct']
    tested = df[df['test_aster_average'].notna()]
    same_session = f'{run_dir}/latency_same_session.csv'
    timing = pd.read_csv(same_session) if os.path.exists(same_session) else df

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), constrained_layout=True)
    ax = axes[0]
    ax.plot(x, df['val_psnr'] - base['val_psnr'], color=INK, linewidth=2, marker='o', markersize=6)
    limit_line(ax, -PSNR_LIMIT, f'Chapter 1 stop rule (-{PSNR_LIMIT} dB)', x.max())
    style(ax, 'a) Validation PSNR change', 'change in PSNR (dB)')

    ax = axes[1]
    for rec, (label, color) in READERS.items():
        y = tested[f'test_{rec}_average'] - base[f'test_{rec}_average']
        ax.plot(tested['removed_pct'], y, color=color, linewidth=2, marker='o', markersize=6, label=label)
        ax.annotate(label, (tested['removed_pct'].iloc[-1], y.iloc[-1]), xytext=(6, 0),
                    textcoords='offset points', va='center', color=INK, fontsize=9)
    limit_line(ax, -OCR_LIMIT, f'-{OCR_LIMIT:g} point', tested['removed_pct'].max())
    style(ax, 'b) Test accuracy change (weighted easy/medium/hard)', 'change in accuracy (points)')
    ax.legend(frameon=False, loc='lower left')

    ax = axes[2]
    ax.plot(x, df['macs'] / base['macs'], color=MUTED, linestyle='--', linewidth=2, marker='s', markersize=6,
            label='MACs')
    for key, (label, color, marker) in LATENCY.items():
        column = f'latency_{key}_ms'
        if column in timing:
            ax.plot(timing['removed_pct'], timing[column] / timing[column].iloc[0], color=color, linewidth=2,
                    marker=marker, markersize=6, label=label)
    source = 'same session' if timing is not df else 'during the run'
    style(ax, f'c) Cost relative to unpruned (latency {source})', 'fraction of the unpruned model')
    ax.legend(frameon=False, loc='lower left')

    for ext in ('png', 'pdf'):
        fig.savefig(f'{run_dir}/figures/rq1_trajectory.{ext}', dpi=200)
    plt.show()

    print(os.path.basename(run_dir))
    for name, column, limit in (('PSNR rule (val drop <= 0.2 dB)', 'val_psnr', PSNR_LIMIT),
                                ('OCR rule  (val CRNN drop <= 1 point)', 'val_crnn', OCR_LIMIT)):
        row = df[df['step'] == kept_step(df, column, limit)].iloc[0]
        test = (' / '.join(f"{row[f'test_{r}_average']:.1f}" for r in READERS)
                if pd.notna(row['test_aster_average']) else 'not tested')
        print(f"  {name}: keeps step {int(row['step'])} ({row['removed_pct']:.1f}% channels removed, "
              f"{int(row['params']):,} params, {row['macs'] / 1e6:.0f} MMACs), test ASTER/MORAN/CRNN {test}")

## 8. Same-session latency (for the paper)

Times every model of each trajectory run back to back in this session (3 interleaved rounds, median), so all numbers come from one GPU and one CPU. Step 0 is the unpruned model each run started from (its `config.json`). Writes `latency_same_session.csv` in the run folder; re-run section 7 afterwards to use it in panel c. Run it on a **T4**, like the existing numbers, so all latencies in the paper come from the same GPU.

In [ ]:
import glob

REDO = False  # True re-times runs that already have latency_same_session.csv
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
for run_dir in sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/*/')):
    if REDO or not os.path.exists(f'{run_dir}/latency_same_session.csv'):
        !python measure_trajectory_latency.py --run_dir {run_dir}

## 9. Re-score every model with the three readers (RQ2, McNemar)

During the runs, validation accuracy used CRNN only. This scores every saved model of the clean runs again, image by image, with ASTER, MORAN and CRNN on the 868 validation and 4,373 test images:

- the **average of the three readers on validation** is a steadier signal for the OCR stop rule;
- **McNemar tests** on the test images say whether each pruned model reads significantly fewer words than the unpruned one.

Writes `rescore/` (one file per model, `summary.csv`) in each run folder. About 2 minutes per model (T4 or L4), roughly 45 minutes for 3 seeds. **If Colab disconnects:** run the setup cells and this cell again; models already scored are skipped.

In [ ]:
import glob

EXPERIMENTS = f'{STORE}/experiments'
RUN_DIRS = sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/tsrn-clean_*/'))
for run_dir in RUN_DIRS:
    print(run_dir)

%cd {REPO}/src
RUN_DIRS_ARG = ' '.join(RUN_DIRS)
!python rescore_trajectory.py --run_dirs {RUN_DIRS_ARG}

## 10. Example figure (for the paper)

Shows a few test images that the unpruned model reads right and the 87.5% model reads wrong (CRNN): input, output at 0% / 50% / 87.5% removed with what CRNN reads and the PSNR, and the target. Needs section 9 to have run. About 2 minutes, any GPU. Saved as `figures/examples_crnn.png/.pdf` in the run folder.

In [ ]:
import glob
from IPython.display import Image, display

EXPERIMENTS = f'{STORE}/experiments'
RUN_DIR = sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/tsrn-clean_*_seed0/'))[0]
%cd {REPO}/src
!python make_examples.py --run_dir {RUN_DIR} --steps 0 4 7 --reader crnn --per_split 2
display(Image(f'{RUN_DIR}/figures/examples_crnn.png'))

## 11. Reading-aware pruning pilot (journal step 3)

One cut of 50% of the channels of the clean TSRN, then fine-tuning (up to 20 epochs, early stop 3), in four versions that differ only in where the CRNN reading loss (CTC, weight 1, at the scale used by STNet: 100 × image loss + 1 × reading loss) is used:

- **A**: image loss only (reference, as in the conference runs);
- **B**: reading-aware channel scoring (Taylor on image + reading loss), image-only fine-tuning;
- **D**: reading-aware scoring and fine-tuning (the full method);
- **C**: image-only scoring, reading-aware fine-tuning (shows how much comes from fine-tuning alone).

Same seed, same scoring batches (8 per arm) and same batch order in every arm. Each arm takes about 20-25 minutes on an L4 (about 1.5 hours in all); an arm that already finished is skipped, so after a disconnect just run the setup cells and this cell again.

**Go** if B or D reads clearly better than A with CRNN (about +0.5 points or more, McNemar p < 0.05) without hurting ASTER and MORAN.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'

REC = 1.0  # weight of the CRNN reading loss
ARMS = {'A_image': (0, 0), 'B_score-ctc': (REC, 0), 'D_score+ft-ctc': (REC, REC), 'C_ft-ctc': (0, REC)}
COMMON = '--ratio 0.5 --round_to 8 --taylor_batches 8 --epochs 20 --patience 3 --min_delta 0.01 --seed 0'
PILOT_ROOT = f'{EXPERIMENTS}/pilot_reading_aware/tsrn-clean_cut50pct_round8_tb8_ft20ep-es3_seed0'

PILOT_DIRS = []
for arm, (score_w, ft_w) in ARMS.items():
    out_dir = f'{PILOT_ROOT}/{arm}_score{score_w:g}_ft{ft_w:g}'
    PILOT_DIRS.append(out_dir)
    print(f'===== {arm}')
    !python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {out_dir} {COMMON} --score_rec_weight {score_w} --ft_rec_weight {ft_w}

DIRS_ARG = ' '.join(PILOT_DIRS)
!python compare_pilots.py --dirs {DIRS_ARG}

## 12. Controls for the reading-aware pilot

Three short runs, same seed and same settings as section 11 (needs section 11 to have run):

- **E**: uncut TSRN, fine-tuned with the image loss only (`--ratio 0`). Extra fine-tuning at a lower learning rate can improve a model by itself.
- **F**: uncut TSRN, fine-tuned with the image + reading loss. Does the reading loss also lift the uncut model?
- **G**: version A (50% cut, image loss) with a fixed 20 epochs (`--patience 0`), the same training time as C.

Then three comparisons (McNemar on the 4,373 test images):

1. **Equal budget:** G (reference) vs A, C, D. Is C still better than A when A trains as long?
2. **Uncut controls:** E (reference) vs F and C. How much does the reading loss give the uncut model?
3. **Price of pruning:** F (reference) vs C and D. Cut vs uncut, both with reading-aware fine-tuning.

About 10–15 minutes per run on an A100 (about 40 minutes in all). A finished run is skipped, so after a disconnect run the setup cells and this cell again.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'

BASE = '--round_to 8 --taylor_batches 8 --epochs 20 --min_delta 0.01 --seed 0 --score_rec_weight 0'
CUT_ROOT = f'{EXPERIMENTS}/pilot_reading_aware/tsrn-clean_cut50pct_round8_tb8_ft20ep-es3_seed0'  # section 11
UNCUT_ROOT = f'{EXPERIMENTS}/pilot_reading_aware/tsrn-clean_cut0pct_ft20ep-es3_seed0'

RUNS = {  # folder: extra arguments
    f'{UNCUT_ROOT}/E_image_score0_ft0': '--ratio 0 --patience 3 --ft_rec_weight 0',
    f'{UNCUT_ROOT}/F_ft-ctc_score0_ft1': '--ratio 0 --patience 3 --ft_rec_weight 1',
    f'{CUT_ROOT}/G_image-fixed20ep_score0_ft0': '--ratio 0.5 --patience 0 --ft_rec_weight 0',
}
for out_dir, extra in RUNS.items():
    print(f'===== {out_dir}')
    !python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {out_dir} {BASE} {extra}

A, C, D = (f'{CUT_ROOT}/{name}' for name in ('A_image_score0_ft0', 'C_ft-ctc_score0_ft1', 'D_score+ft-ctc_score1_ft1'))
E, F = f'{UNCUT_ROOT}/E_image_score0_ft0', f'{UNCUT_ROOT}/F_ft-ctc_score0_ft1'
G = f'{CUT_ROOT}/G_image-fixed20ep_score0_ft0'
OUT = f'{EXPERIMENTS}/pilot_reading_aware'
print()
print('===== 1. equal budget: G vs A, C, D')
!python compare_pilots.py --dirs {G} {A} {C} {D} --out {OUT}/comparison_1_equal_budget.csv
print()
print('===== 2. uncut controls: E vs F, C')
!python compare_pilots.py --dirs {E} {F} {C} --out {OUT}/comparison_2_uncut_controls.csv
print()
print('===== 3. price of pruning with reading-aware fine-tuning: F vs C, D')
!python compare_pilots.py --dirs {F} {C} {D} --out {OUT}/comparison_3_price_of_pruning.csv

## 13. Confirmation: uncut model, reading retraining, fixed 20 epochs

In section 12, the uncut model with reading-aware fine-tuning (F) was stopped by the PSNR early stop after 9 epochs while its reading was still improving, whereas the cut model (C) trained 20 epochs. **H** is F with a fixed 20 epochs (`--patience 0`), so the price of pruning can be measured at equal training time.

Then two comparisons (McNemar on the 4,373 test images):

1. **Price of pruning at equal training time:** H (uncut, reference) vs C and D (cut 50%).
2. **Reading loss on the uncut model:** E (uncut, image loss) vs F and H.

About 15 minutes on an L4 or A100. If it already finished, the run is skipped and only the comparisons are printed.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'

CUT_ROOT = f'{EXPERIMENTS}/pilot_reading_aware/tsrn-clean_cut50pct_round8_tb8_ft20ep-es3_seed0'
UNCUT_ROOT = f'{EXPERIMENTS}/pilot_reading_aware/tsrn-clean_cut0pct_ft20ep-es3_seed0'
H = f'{UNCUT_ROOT}/H_ft-ctc-fixed20ep_score0_ft1'
!python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {H} --ratio 0 --round_to 8 --taylor_batches 8 --epochs 20 --patience 0 --min_delta 0.01 --seed 0 --score_rec_weight 0 --ft_rec_weight 1

C, D = f'{CUT_ROOT}/C_ft-ctc_score0_ft1', f'{CUT_ROOT}/D_score+ft-ctc_score1_ft1'
E, F = f'{UNCUT_ROOT}/E_image_score0_ft0', f'{UNCUT_ROOT}/F_ft-ctc_score0_ft1'
OUT = f'{EXPERIMENTS}/pilot_reading_aware'
print()
print('===== 4. price of pruning at equal training time: H vs C, D')
!python compare_pilots.py --dirs {H} {C} {D} --out {OUT}/comparison_4_price_of_pruning_20ep.csv
print()
print('===== 5. reading loss on the uncut model: E vs F, H')
!python compare_pilots.py --dirs {E} {F} {H} --out {OUT}/comparison_5_uncut_reading_loss.csv

## 14. Fair 2 x 2: model size x fine-tuning loss, same protocol

Picking the best epoch by validation PSNR favoured the cut model: its PSNR rises while it recovers, so late epochs are kept, but the uncut model's PSNR is flat, so an early epoch is kept (run H kept epoch 2 of 20 and threw away most of the reading training). Here every run trains exactly 20 epochs and **keeps the last epoch** (`--select_by last`), with no early stop and no selection:

| | image loss | image + reading loss |
|---|---|---|
| **uncut** | L | J |
| **cut 50%** | K | I |

Then the McNemar comparisons:

1. **Price of pruning with reading retraining:** J vs I.
2. **Price of pruning with image retraining:** L vs K.
3. **Effect of the reading loss:** K vs I (cut) and L vs J (uncut).

About 15 minutes per run on an L4 (about 1 hour in all). Finished runs are skipped after a disconnect.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'

ROOT = f'{EXPERIMENTS}/pilot_reading_aware/fair_2x2_ft20ep-last_tb8_seed0'
COMMON = '--round_to 8 --taylor_batches 8 --epochs 20 --patience 0 --select_by last --seed 0 --score_rec_weight 0'
RUNS = {  # name: (cut ratio, reading-loss weight)
    'I_cut50_reading': (0.5, 1), 'J_uncut_reading': (0, 1), 'K_cut50_image': (0.5, 0), 'L_uncut_image': (0, 0),
}
for name, (ratio, rec) in RUNS.items():
    print(f'===== {name}')
    !python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {ROOT}/{name} {COMMON} --ratio {ratio} --ft_rec_weight {rec}

I, J, K, L = (f'{ROOT}/{name}' for name in RUNS)
print()
print('===== 1. price of pruning with reading retraining: J (uncut) vs I (cut)')
!python compare_pilots.py --dirs {J} {I} --out {ROOT}/comparison_price_reading.csv
print()
print('===== 2. price of pruning with image retraining: L (uncut) vs K (cut)')
!python compare_pilots.py --dirs {L} {K} --out {ROOT}/comparison_price_image.csv
print()
print('===== 3a. reading loss on the cut model: K vs I')
!python compare_pilots.py --dirs {K} {I} --out {ROOT}/comparison_reading_cut.csv
print()
print('===== 3b. reading loss on the uncut model: L vs J')
!python compare_pilots.py --dirs {L} {J} --out {ROOT}/comparison_reading_uncut.csv

## 15. Heavy cut: does the reading loss help more when pruning hurts more?

At a 50% cut, the reading loss helped the uncut model (J vs L) at least as much as the cut model (I vs K), so it is a general training gain, not a pruning repair. Pruning-specific recovery should show up where pruning really hurts: at 87.5% removed (CRNN about -4.6 in the step-by-step runs).

Same protocol as section 14 (20 epochs, last epoch kept), one cut of 87.5%:

| | image loss | image + reading loss |
|---|---|---|
| **cut 87.5%** | N | M |

Then the McNemar comparisons:

1. **Reading loss on the heavy cut:** N vs M (compare with L vs J and K vs I).
2. **Price of pruning with image retraining:** L vs N.
3. **Price of pruning with reading retraining:** J vs M.

About 15 minutes per run on an L4 (about 30 minutes in all). Needs the section 14 runs J and L.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'

ROOT = f'{EXPERIMENTS}/pilot_reading_aware/fair_2x2_ft20ep-last_tb8_seed0'
COMMON = '--round_to 8 --taylor_batches 8 --epochs 20 --patience 0 --select_by last --seed 0 --score_rec_weight 0'
RUNS = {'M_cut88_reading': (0.875, 1), 'N_cut88_image': (0.875, 0)}  # name: (cut ratio, reading-loss weight)
for name, (ratio, rec) in RUNS.items():
    print(f'===== {name}')
    !python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {ROOT}/{name} {COMMON} --ratio {ratio} --ft_rec_weight {rec}

M, N = (f'{ROOT}/{name}' for name in RUNS)
J, L = f'{ROOT}/J_uncut_reading', f'{ROOT}/L_uncut_image'
print()
print('===== 1. reading loss on the 87.5% cut: N vs M')
!python compare_pilots.py --dirs {N} {M} --out {ROOT}/comparison_reading_cut88.csv
print()
print('===== 2. price of the 87.5% cut with image retraining: L vs N')
!python compare_pilots.py --dirs {L} {N} --out {ROOT}/comparison_price88_image.csv
print()
print('===== 3. price of the 87.5% cut with reading retraining: J vs M')
!python compare_pilots.py --dirs {J} {M} --out {ROOT}/comparison_price88_reading.csv

## 16. GRU pruning: is the model really faster, and what does reading cost?

Cutting only convolutions gives at most about x1.5 real speed-up, because the GRUs (72% of single-image CPU time) are untouched. Here the GRUs are cut too: whole hidden units (all 3 gates) together with the 64-channel trunk they feed, plus the GRU input features (`--gru_ratio`).

First a self-check on random weights (CPU, a few seconds): it must end with **All checks passed**.

Then two runs, same protocol as section 14 (20 epochs, last epoch kept), all with the reading loss:

| Run | conv cut | GRU cut |
|---|---|---|
| J (section 14) | 0 | 0 |
| I (section 14) | 50% | 0 |
| **O** | 50% | 50% |
| **P** | 0 | 50% |

Then McNemar vs the uncut model J, and the real speed-up of each run (unpruned / pruned time, measured in the same session). About 15 minutes per run on an L4.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
!python check_pruning.py
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'

PAIRS = f'{EXPERIMENTS}/pilot_reading_aware/fair_2x2_ft20ep-last_tb8_seed0'
ROOT = f'{EXPERIMENTS}/pilot_gru/ft20ep-last_tb8_reading_seed0'
COMMON = '--round_to 8 --taylor_batches 8 --epochs 20 --patience 0 --select_by last --seed 0 --score_rec_weight 0 --ft_rec_weight 1'
RUNS = {'O_conv50_gru50': (0.5, 0.5), 'P_gru50': (0, 0.5)}  # name: (conv cut, GRU cut)
for name, (ratio, gru_ratio) in RUNS.items():
    print(f'===== {name}')
    !python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {ROOT}/{name} {COMMON} --ratio {ratio} --gru_ratio {gru_ratio}

import json
J, I = f'{PAIRS}/J_uncut_reading', f'{PAIRS}/I_cut50_reading'
O, P = (f'{ROOT}/{name}' for name in RUNS)
print()
print('===== reading accuracy vs the uncut model J (all with reading-loss fine-tuning)')
!python compare_pilots.py --dirs {J} {I} {O} {P} --out {ROOT}/comparison_vs_uncut.csv
print()
print('===== real speed-up (unpruned time / pruned time, same session)')
for name, path in {'I conv 50%': I, 'O conv 50% + GRU 50%': O, 'P GRU 50%': P}.items():
    stages = json.load(open(f'{path}/results.json'))['stages']
    before, after = stages['baseline']['latency_ms'], stages['pruned + fine-tune']['latency_ms']
    print(f'{name:22s}' + '  '.join(f'{key} x{before[key] / after[key]:.2f}' for key in ('gpu_b1', 'gpu_b64', 'cpu1_b1')))

## 17. Depth vs width: which cuts make TSRN really faster? (no training, about 5 minutes)

Thinner layers (width pruning, sections 14 to 16) barely changed the time for one image. Here whole blocks or whole GRUs are removed instead (depth pruning), and all options are timed in the same session as the width cuts. Speed does not depend on the weights, so random weights are used and nothing is trained.

**Use an L4**, the GPU of all the paper's timings.

1. Self-check (seconds): it must end with **All checks passed**.
2. Timing of 14 options in 3 interleaved rounds: size, GRU steps, GPU batch 1 / batch 64 and CPU time, speed-up vs the uncut model.
3. A fit that shows what sets the time: the GRU steps that run one after another, or the amount of computation (MACs).

Results are saved in `experiments/depth_latency/`.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
!python check_pruning.py
!python measure_depth_latency.py --out_dir {EXPERIMENTS}/depth_latency

## 18. Depth pruning: how much reading do the fast options lose?

Section 17 showed that only removing depth makes one image faster. Here the three most useful options are pruned from the trained model and fine-tuned, with the same protocol as sections 14 to 16 (reading loss, 20 epochs, last epoch kept):

| Run | What is removed | Speed-up GPU / CPU, 1 image (section 17) |
|---|---|---|
| **Q** | 2 of the 5 blocks (3 left) | x1.6 / x1.5 |
| **R** | 3 of the 5 blocks (2 left) | x2.2 / x2.0 |
| **S** | the horizontal GRU of every block | x1.5 / x1.9 |

**Which blocks go is chosen by reading:** the script tries removing each block, keeps the removal that leaves validation CRNN accuracy highest, and repeats.

Then McNemar vs the uncut model J (section 14) and the speed-up measured in the same session. **Use an L4.** About 15 minutes per run (about 45 minutes in all). Finished runs are skipped after a disconnect.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'

PAIRS = f'{EXPERIMENTS}/pilot_reading_aware/fair_2x2_ft20ep-last_tb8_seed0'
ROOT = f'{EXPERIMENTS}/pilot_depth/ft20ep-last_tb8_reading_select-crnn_seed0'
COMMON = '--ratio 0 --gru_ratio 0 --round_to 8 --taylor_batches 8 --epochs 20 --patience 0 --select_by last --seed 0 --score_rec_weight 0 --ft_rec_weight 1 --depth_criterion crnn'
RUNS = {'Q_3blocks': '--remove_blocks 2', 'R_2blocks': '--remove_blocks 3', 'S_no_horizontal_grus': '--remove_grus gru2'}
for name, depth in RUNS.items():
    print(f'===== {name}')
    !python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {ROOT}/{name} {COMMON} {depth}

import json
J = f'{PAIRS}/J_uncut_reading'
Q, R, S = (f'{ROOT}/{name}' for name in RUNS)
print()
print('===== reading accuracy vs the uncut model J (all with reading-loss fine-tuning)')
!python compare_pilots.py --dirs {J} {Q} {R} {S} --out {ROOT}/comparison_vs_uncut.csv
print()
print('===== removed parts and real speed-up (unpruned time / pruned time, same session)')
for name in RUNS:
    results = json.load(open(f'{ROOT}/{name}/results.json'))
    before, after = results['stages']['baseline']['latency_ms'], results['stages']['pruned + fine-tune']['latency_ms']
    print(f'{name:22s}' + '  '.join(f'{key} x{before[key] / after[key]:.2f}' for key in ('gpu_b1', 'gpu_b64', 'cpu1_b1'))
          + f"  | removed: {' '.join(results['depth'])}")

## 19. Is depth pruning better than training the shallow model from scratch?

Section 18: **R** (blocks 2, 3 and 4 removed, 2 left) is about x2 faster on one image but reads about 10 CRNN points less than the uncut model J, and its validation reading was still rising at epoch 20. Two runs decide whether pruning is worth it, or whether one should simply train the small model (Liu et al. 2019, arXiv:1810.05270):

| Run | Starting point | Training |
|---|---|---|
| **T** | the same 2-block TSRN (same layers as R) **trained from scratch** with the full recipe (500 epochs, image loss) | then exactly R's fine-tune: 20 epochs, reading loss, full TSRN as teacher |
| **U** | R: the trained full TSRN with blocks 2-4 removed | **100** fine-tuning epochs instead of 20 |

- **T vs R**: same model and same fine-tune, only the starting point differs (pruned from the big model vs trained from scratch).
- **U vs R**: does more retraining close the gap to J?

**Use an A100.** First cell: the from-scratch training, about 1-1.5 hours (the full TSRN took about 2 hours), resumable like section 2b: if Colab disconnects, run the setup cells and this cell again. Second cell: T (about 10 minutes) and U (about 40-60 minutes); a finished run is skipped, but a run cut by a disconnect starts again.

In [ ]:
import json, os
import torch

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
!python check_pruning.py

ROOT18 = f'{EXPERIMENTS}/pilot_depth/ft20ep-last_tb8_reading_select-crnn_seed0'
PARTS = ' '.join(json.load(open(f'{ROOT18}/R_2blocks/results.json'))['depth'])  # the blocks removed in R
big_gpu = torch.cuda.get_device_properties(0).total_memory > 30e9
TRAIN_ARGS = f'--remove_parts {PARTS} --workers {min(8, os.cpu_count())}' + ('' if big_gpu else ' --micro_batch 256')
print(torch.cuda.get_device_name(0), '|', TRAIN_ARGS)
!python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} {TRAIN_ARGS}

In [ ]:
import json, os

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
ROOT18 = f'{EXPERIMENTS}/pilot_depth/ft20ep-last_tb8_reading_select-crnn_seed0'
PARTS = ' '.join(json.load(open(f'{ROOT18}/R_2blocks/results.json'))['depth'])
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'
SHALLOW_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --remove_parts {PARTS} --print_run_dir
SHALLOW_DIR = SHALLOW_DIR[-1]
SHALLOW_CKPT = f'{SHALLOW_DIR}/checkpoints/final.pth'
assert os.path.exists(f'{SHALLOW_DIR}/test_final.json'), 'finish the from-scratch training (cell above) first'

# Same protocol as section 18; only the starting point (T) or the number of epochs (U) changes.
COMMON = ('--ratio 0 --gru_ratio 0 --round_to 8 --taylor_batches 8 --patience 0 --select_by last --seed 0 '
          '--score_rec_weight 0 --ft_rec_weight 1 --depth_criterion crnn')
J = f'{EXPERIMENTS}/pilot_reading_aware/fair_2x2_ft20ep-last_tb8_seed0/J_uncut_reading'
R = f'{ROOT18}/R_2blocks'  # section 18
T = f'{ROOT18}/T_scratch_2blocks'
ROOT100 = f'{EXPERIMENTS}/pilot_depth/ft100ep-last_tb8_reading_select-crnn_seed0'
U = f'{ROOT100}/U_2blocks_100ep'
print('===== T: 2-block TSRN from scratch, then 20 epochs of reading fine-tune with the full TSRN as teacher')
!python prune_tsrn_pilot.py --resume {SHALLOW_CKPT} --teacher {BASELINE_CKPT} --out_dir {T} {COMMON} --epochs 20
print('===== U: R with 100 fine-tuning epochs')
!python prune_tsrn_pilot.py --resume {BASELINE_CKPT} --out_dir {U} {COMMON} --epochs 100 --remove_blocks 3

print()
print('===== vs the uncut model J')
!python compare_pilots.py --dirs {J} {R} {T} {U} --out {ROOT100}/comparison_vs_uncut.csv
print()
print('===== vs R (pruned, 20 epochs): T = from scratch instead of pruned, U = 100 epochs instead of 20')
!python compare_pilots.py --dirs {R} {T} {U} --out {ROOT100}/comparison_vs_R.csv
print()
print('===== architecture and validation CRNN during fine-tuning (every 10 epochs)')
for name, folder in {'R pruned, 20 ep': R, 'T scratch, 20 ep': T, 'U pruned, 100 ep': U}.items():
    results = json.load(open(f'{folder}/results.json'))
    stage = results['stages']['pruned + fine-tune']
    curve = ' '.join(f"{h['val']['accuracy']['crnn']:.1f}" for h in results['fine_tune'][9::10])
    print(f"{name:18s} {stage['params']:,d} params | removed {' '.join(results['depth'])} | val CRNN {curve}")
scratch = json.load(open(f'{SHALLOW_DIR}/test_final.json'))['test']['average']['accuracy']
print('2-block TSRN from scratch, before fine-tuning (image loss, 500 epochs): ASTER/MORAN/CRNN '
      + ' / '.join(f'{scratch[rec]:.2f}' for rec in ('aster', 'moran', 'crnn')) + '  (full TSRN: 55.59 / 53.05 / 41.46)')

## 20. LEMMA as the teacher of the fast 2-block TSRN (kill test)

Goal: make **T** (the 2-block TSRN, about x2 faster than TSRN) read better without slowing it down. **LEMMA** (Guo et al., IJCAI 2023, the best model with released weights) is only *run*, never trained: its output becomes the distillation target in T's usual fine-tune, so nothing is added at inference.

**Use an L4.** First cell, about 15 minutes:
1. Self-check, then LEMMA's released file is downloaded once to Drive (`lemma_release.pth`).
2. LEMMA's test accuracy with our three readers. Its README reports **66.93 / 63.60 / 56.73** (ASTER / MORAN / CRNN); a much lower result means our copy or our evaluation differs.
3. Size, and speed in one session: LEMMA, its guidance branch alone, the full TSRN and T.

Second cell, about 30-45 minutes: the kill test. Same protocol as T (20 epochs, reading loss, last epoch kept); only the teacher changes.

| Run | Teacher | Distillation weight |
|---|---|---|
| **T** (section 19) | full TSRN | 0.1 |
| **V** | LEMMA | 0.1 (as T) |
| **W** | LEMMA | 1 (stronger pull towards LEMMA) |

**Decision rule, fixed before the run:** continue only if V or W reads at least **+2 CRNN points** more than T, with no significant loss on ASTER or MORAN (McNemar p < 0.05). Otherwise LEMMA distillation is dropped.

If the download fails ("too many users"), open the checkpoint link in LEMMA's README (github.com/csguoh/LEMMA) in a browser, save the file to your Drive as `MyDrive/textzoom/lemma_release.pth`, and run the cell again.

In [ ]:
import os

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
!python check_pruning.py

# LEMMA's released model (checkpoint link in the README of github.com/csguoh/LEMMA), downloaded once to Drive.
LEMMA_CKPT = f'{STORE}/lemma_release.pth'
![ -f {LEMMA_CKPT} ] || gdown 1iuVc0fh5rQAT2Ep5KgyV1GnsXPkdI4V0 -O {LEMMA_CKPT}
assert os.path.exists(LEMMA_CKPT), 'download failed: see the note above'

ROOT18 = f'{EXPERIMENTS}/pilot_depth/ft20ep-last_tb8_reading_select-crnn_seed0'
T = f'{ROOT18}/T_scratch_2blocks'
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'
!python evaluate_lemma.py --lemma {LEMMA_CKPT} --compare {BASELINE_CKPT} {T}/tsrn_pruned.pth --compare_names full_TSRN T_2blocks --out_dir {EXPERIMENTS}/lemma

In [ ]:
import csv, json

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
LEMMA_CKPT = f'{STORE}/lemma_release.pth'
ROOT18 = f'{EXPERIMENTS}/pilot_depth/ft20ep-last_tb8_reading_select-crnn_seed0'
PARTS = ' '.join(json.load(open(f'{ROOT18}/R_2blocks/results.json'))['depth'])
SHALLOW_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --remove_parts {PARTS} --print_run_dir
SHALLOW_CKPT = f'{SHALLOW_DIR[-1]}/checkpoints/final.pth'

# Exactly T's protocol (section 19); only the teacher changes (and, in W, the distillation weight).
COMMON = ('--ratio 0 --gru_ratio 0 --round_to 8 --taylor_batches 8 --patience 0 --select_by last --seed 0 '
          '--score_rec_weight 0 --ft_rec_weight 1 --depth_criterion crnn --epochs 20')
T = f'{ROOT18}/T_scratch_2blocks'
V = f'{ROOT18}/V_scratch_2blocks_lemma-kd0.1'
W = f'{ROOT18}/W_scratch_2blocks_lemma-kd1'
print('===== V: LEMMA as teacher, distillation weight 0.1 (as T)')
!python prune_tsrn_pilot.py --resume {SHALLOW_CKPT} --teacher_lemma {LEMMA_CKPT} --out_dir {V} {COMMON}
print('===== W: LEMMA as teacher, distillation weight 1')
!python prune_tsrn_pilot.py --resume {SHALLOW_CKPT} --teacher_lemma {LEMMA_CKPT} --out_dir {W} {COMMON} --kd_weight 1

print()
print('===== vs T (full TSRN as teacher)')
OUT = f'{ROOT18}/comparison_lemma_teacher_vs_T.csv'
!python compare_pilots.py --dirs {T} {V} {W} --out {OUT}

print()
print('===== decision rule: at least +2 CRNN vs T, and no significant loss (p < 0.05) on ASTER or MORAN')
rows = list(csv.DictReader(open(OUT)))
for row in rows[1:]:
    gain = {rec: float(row[f'test_{rec}']) - float(rows[0][f'test_{rec}']) for rec in ('aster', 'moran', 'crnn')}
    hurt = [rec for rec in ('aster', 'moran') if gain[rec] < 0 and float(row[f'{rec}_p_vs_ref']) < 0.05]
    verdict = 'PASS' if gain['crnn'] >= 2 and not hurt else 'FAIL'
    print(f"{row['pilot']}: " + ', '.join(f"{rec.upper()} {g:+.2f} (p={float(row[rec + '_p_vs_ref']):.3g})"
                                           for rec, g in gain.items()) + f'  -> {verdict}')

## 21. LEMMA with a small guidance network (kill test)

Section 20: LEMMA reads best (67.4 / 65.0 / 56.5 in our pipeline), but its text-guidance branch (an ABINet-style recognizer plus alignment transformers: 36.5M of its 37.9M parameters) takes 54-74% of its time. That branch only makes a 64 x 16 x 64 guidance map for the image branch. Here it is **replaced** by GuidanceNet, a small convolutional network trained only to reproduce LEMMA's guidance map and LEMMA's output. LEMMA's image branch keeps its released weights, frozen. No reader is used in training, so all three test readers are unseen.

| Run | GuidanceNet | Parameters |
|---|---|---|
| width 64 | 64 channels, 5 dilated residual blocks | about 0.4M |
| width 128 | 128 channels, 5 dilated residual blocks | about 1.6M |

**Decision rule, fixed before the run:** PASS if a student loses **at most 2 points against LEMMA** on every reader (our LEMMA: 67.35 / 65.04 / 56.48, ASTER / MORAN / CRNN) **and** is at least **x2 faster than LEMMA** on CPU and on GPU batch 64 (where the guidance takes 72% and 74% of LEMMA's time; GPU batch 1 is reported). Otherwise the LEMMA line is dropped.

Why this bar: 2 points below LEMMA still reads above every other model in SGENet's Table 1 (best: TATT 63.6 / 59.5 / 52.6). Merely matching SGENet (61.4 / 57.3 / 49.0) is not enough: the student keeps LEMMA's 5-block image branch, so it costs more than SGENet, and SGENet would be the better deal.

**Use an L4** (the GPU of section 20). Needs section 20's first cell (LEMMA's file and `lemma_results.json`). About 35-40 minutes per run, because LEMMA runs as the teacher on every batch. A finished run is skipped; a run interrupted after training reuses its saved model (only testing and timing run again).

In [ ]:
import json, os

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
!python check_pruning.py

LEMMA_CKPT = f'{STORE}/lemma_release.pth'
LEMMA_RESULTS = f'{EXPERIMENTS}/lemma/lemma_results.json'
assert os.path.exists(LEMMA_CKPT) and os.path.exists(LEMMA_RESULTS), "run section 20's first cell first"
ROOT18 = f'{EXPERIMENTS}/pilot_depth/ft20ep-last_tb8_reading_select-crnn_seed0'
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = f'{CLEAN_DIR[-1]}/checkpoints/final.pth'
COMPARE = f'--compare {BASELINE_CKPT} {ROOT18}/T_scratch_2blocks/tsrn_pruned.pth --compare_names full_TSRN T_2blocks'

ROOT21 = f'{EXPERIMENTS}/lemma_fast'
for width in (64, 128):
    print(f'===== GuidanceNet width {width}')
    !python distill_lemma_guidance.py --lemma {LEMMA_CKPT} --lemma_results {LEMMA_RESULTS} --width {width} --out_dir {ROOT21}/width{width}_ep20_seed0 {COMPARE}

print()
print('===== summary')
for width in (64, 128):
    decision = json.load(open(f'{ROOT21}/width{width}_ep20_seed0/results.json'))['decision']
    print(f'width {width}: ASTER / MORAN / CRNN '
          + ' / '.join(f"{decision['accuracy'][rec]:.2f}" for rec in ('aster', 'moran', 'crnn'))
          + ' | speed-up vs LEMMA ' + ', '.join(f'{key} x{value:.2f}' for key, value in decision['speedup_vs_lemma'].items())
          + f" -> {decision['verdict']}")

## 22. Checks before a small reader for LEMMA (no training)

Section 21 failed because a network that never saw a letter cannot copy LEMMA's guidance. The next step replaces only LEMMA's **reader** (its ABINet vision and language models: 36.2M parameters, 97% of the guidance computation) with a small reader trained on letter labels, and keeps LEMMA's alignment and image branch as released. The spec is in `project/lemma_reader_spec.md`.

These four checks run LEMMA's released model with parts of its reader timed, removed or replaced. They decide what the small reader must contain:

| Check | What runs | What it tells us |
|---|---|---|
| C1 | each part of the reader and the alignment, timed in one session with LEMMA (GPU batch 1 and 64, CPU batch 1) | which part to shrink first |
| C2 | LEMMA guided by the vision part only (no language model) | whether the small reader needs a language model |
| C3 | the reader's own word accuracy on the LR test images | the reading level the small reader must approach |
| C4 | LEMMA guided by the true words (one-hot letters, true lengths, the reader's own maps) | how much the final accuracy depends on reading |

**Rules fixed before the run:**
- **C1:** the part with the largest share of the guidance time on CPU batch 1 and GPU batch 64 is the first to replace.
- **C2:** within 1 point of LEMMA on every reader means the small reader needs no language model.
- **C3 + C4:** test points per point of reading accuracy, (C4 − LEMMA) / (100 − C3), gives how much reading accuracy the small reader may lose within the 2-point budget. This is a rough, straight-line estimate.

**Use an L4**, the GPU of section 20, so the times match. Needs section 20's first cell (LEMMA's file and `lemma_results.json`). About 10–15 minutes. The results are saved after every check, and a rerun skips the checks already done.

In [ ]:
import os

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
!python check_pruning.py

LEMMA_CKPT = f'{STORE}/lemma_release.pth'
LEMMA_RESULTS = f'{EXPERIMENTS}/lemma/lemma_results.json'
assert os.path.exists(LEMMA_CKPT) and os.path.exists(LEMMA_RESULTS), "run section 20's first cell first"
!python check_lemma_reader.py --lemma {LEMMA_CKPT} --lemma_results {LEMMA_RESULTS} --out_dir {EXPERIMENTS}/lemma_reader_checks